# Локальное развёртывание и мониторинг

Этот ноутбук показывает сохранённые технические проверки. Run All не запускает Docker, нагрузку или банковский holdout. Стенд использует демонстрационные статьи и lexical поиск. Банковский корпус research_only в него не переносится.

Для полной конфигурации с мониторингом освободите минимум 4 GiB на диске перед запуском. Тестовый прогон на Windows столкнулся с заполнением C и ошибкой ввода-вывода Docker; статус проверки нужно смотреть в отчёте и CI, а не считать запуск успешным по наличию конфигурационных файлов.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
REPORT = ROOT / 'reports/local_operations.json'
if REPORT.exists():
    report = json.loads(REPORT.read_text(encoding='utf-8'))
    display(pd.Series({key: value for key, value in report.items() if key != 'deployment'}))
else:
    print('Сводный отчёт ещё не опубликован.')


## Воспроизводимая проверка

```powershell
docker build -t x20-support-api:latest .
python scripts/verify_local_deployment.py --monitoring
```

Скрипт создаёт отдельный Compose project, случайные ключи и свободные loopback порты. Проверяет HTTPS с явно переданным CA, отсутствие доверия без CA, 401, цитату, отказ, параллельные запросы и 429. Затем проверяет сбор Prometheus, загрузку dashboard и alert rules. В конце удаляет только свои контейнеры, volumes и временные секреты. Системное хранилище сертификатов не меняется.

В GitHub Actions есть отдельный Linux job для этой проверки и jobs Windows/Linux для тестов и ноутбуков. Наличие YAML не означает успешного CI: нужен фактический зелёный запуск.

In [ ]:
paths = sorted((ROOT / 'work/local_deployment').glob('*/validation.json'))
if paths:
    latest = json.loads(paths[-1].read_text(encoding='utf-8'))
    display(pd.Series(latest['checks']))
    display(pd.DataFrame(latest['search_profiles']))
else:
    print('Локально завершённого нагрузочного отчёта пока нет.')


## Постоянный локальный стенд

```powershell
python scripts/manage_local_stack.py up --monitoring
python scripts/manage_local_stack.py status
python scripts/manage_local_stack.py down
```

Сначала соберите образ. HTTPS доступен на https://localhost:8443, Grafana на http://127.0.0.1:3000. Секреты находятся только в ignored work/local_stack: api_key и grafana_password. Пароль Grafana относится к пользователю admin. Не публикуйте эти файлы и не вставляйте ключ в ноутбук. Команда down сохраняет данные мониторинга в volumes.

Для HTTP-клиента передайте work/local_stack/root.crt явно. Браузер без доверия к этому CA покажет предупреждение; скрипт не устанавливает доверие в Windows. API напрямую на порт хоста не публикуется. Prometheus и Grafana доступны только на loopback. Для внешнего сервера нужны отдельные DNS, TLS и правила доступа.

## Как интерпретировать нагрузку

Клиент выполняет закрытый цикл: каждый worker начинает следующий запрос после предыдущего. В профилях 1, 4 и 12 workers по 120 запросов; дополнительно есть 10 секунд с 8 workers. Каждое обращение создаёт новое HTTPS соединение. Клиент подключается к IPv4 loopback с проверкой TLS имени localhost, без системного proxy и задержки IPv6 fallback.

Отдельный профиль лимита использует 60 запросов при лимите 20: ожидаются 20 ответов 200 и 40 ответов 429. Ожидаемые 429 учитываются как успешная проверка ограничения, но не считаются успешными поисковыми запросами.

P50/P95/P99 клиента включают TLS, транспорт, API и чтение ответа. Гистограмма API измеряет только серверную обработку до возврата response; она не равна клиентскому round-trip. Десять секунд не доказывают устойчивость за сутки. LLM, агент и классификатор в эти нагрузочные профили не входят. SLA не установлен.

## Дашборд и оповещения

Grafana показывает скорость запросов, приближённый P95 серверной задержки по histogram buckets, отказы 401/429 и наличие индекса/модели. Counter метрики сбрасываются при перезапуске процесса. Текущий rate limit и счётчики относятся к одному API process; добавление нескольких workers требует общей реализации лимита и корректной агрегации.

Prometheus содержит четыре правила: недоступный scrape target, отсутствие индекса, повышенная доля 5xx и частые 429. Правила используют стартовые исследовательские пороги. Alertmanager и отправка сообщений не настроены: внешние уведомления не отправляются. Секрет scrape читается из отдельного файла.

При обновлении исходников повторите сборку и проверки. Для отката запускайте ранее проверенный image ID через --image. Обновление банковской базы и снятие карантина требуют отдельного протокола версий, проверки источников и экспертной приёмки.